# Yambda 50M likes — CDN-style Two-Tower baseline

## 0. Imports and config

In [1]:
import os
import json
import random
import itertools
import gc
from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd
import polars as pl

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from tqdm import tqdm


def set_seed(seed: int = 0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


CFG = {
    # Paths
    "data_dir": "../data",
    "interactions_file": "likes.parquet",
    "artist_file": "artist_item_mapping.parquet",
    "album_file": "album_item_mapping.parquet",

    # Debug/data filtering
    # None = all users; for debugging set 50_000 or 100_000
    "max_users": None,
    "min_user_interactions": 3,
    "min_item_interactions": 5,

    # Model
    "embed_dim": 64,
    "artist_emb_dim": 32,
    "album_emb_dim": 32,
    "tower_hidden": [256, 128, 64],
    "dropout": 0.0,

    # Training
    "batch_size": 4096,
    "grad_clip": 1.0,
    "lr": 1e-3,
    "weight_decay": 0.0,

    # Grid search
    "tune_fast": True,
    "tune_epochs": 3,
    "tune_val_fraction": 1.00,
    "skip_tune_if_cached": True,
    "cache_path": "best_hparams_yambda_likes_features.json",

    # Final training
    "final_epochs": 20,

    # Evaluation
    "eval_k": [10, 50],
    "eval_batch_users": 128,
    "eval_item_batch": 8192,
    "head_fraction": 0.20,

    # Reproducibility
    "seed": 0,
    "seeds": [0, 1, 2, 3, 4],
}

set_seed(CFG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print(f"Final: {len(CFG['seeds'])} seed(s) × {CFG['final_epochs']} epochs")


device: cuda
Final: 5 seed(s) × 20 epochs


## 1. Load data

In [2]:
def find_file(data_dir: str | Path, name: str) -> Path:
    data_dir = Path(data_dir)
    candidates = [
        data_dir / name,
        data_dir / f"{name}.parquet",
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(f"Could not find {name}. Tried: {candidates}")


def normalize_interaction_columns(df: pl.DataFrame) -> pl.DataFrame:
    rename = {}
    for c in df.columns:
        lc = c.lower()
        if lc in {"uid", "user_id", "userid", "user"}:
            rename[c] = "uid"
        elif lc in {"item_id", "itemid", "track_id", "trackid", "song_id"}:
            rename[c] = "item_id"
        elif lc in {"timestamp", "ts", "time", "event_timestamp", "datetime"}:
            rename[c] = "timestamp"
    return df.rename(rename)


def normalize_item_side_columns(df: pl.DataFrame) -> pl.DataFrame:
    rename = {}
    for c in df.columns:
        lc = c.lower()
        if lc in {"item_id", "itemid", "track_id", "trackid"}:
            rename[c] = "item_id"
        elif lc in {"artist_id", "artistid"}:
            rename[c] = "artist_id"
        elif lc in {"album_id", "albumid"}:
            rename[c] = "album_id"
    return df.rename(rename)


DATA_DIR = Path(CFG["data_dir"])
INTERACTIONS_PATH = find_file(DATA_DIR, CFG["interactions_file"])
ARTIST_PATH = find_file(DATA_DIR, CFG["artist_file"])
ALBUM_PATH = find_file(DATA_DIR, CFG["album_file"])

print("interactions:", INTERACTIONS_PATH)
print("artists:", ARTIST_PATH)
print("albums:", ALBUM_PATH)

interactions = pl.read_parquet(INTERACTIONS_PATH)
interactions = normalize_interaction_columns(interactions)

print("raw interactions:", interactions.shape)
print("columns:", interactions.columns)
print(interactions.head())

required = {"uid", "item_id"}
missing = required - set(interactions.columns)
assert not missing, f"Missing required columns {missing}. Available: {interactions.columns}"

if "timestamp" not in interactions.columns:
    print("[WARN] timestamp not found: using row index as timestamp")
    interactions = interactions.with_row_index("timestamp")

interactions = interactions.select(["uid", "item_id", "timestamp"])

# Deduplicate repeated likes by the same user for the same item.
interactions = (
    interactions
    .sort("timestamp")
    .unique(subset=["uid", "item_id"], keep="first")
)

print("after dedup:", interactions.shape)

interactions: ../data/likes.parquet
artists: ../data/artist_item_mapping.parquet
albums: ../data/album_item_mapping.parquet
raw interactions: (881456, 4)
columns: ['uid', 'timestamp', 'item_id', 'is_organic']
shape: (5, 4)
┌─────┬───────────┬─────────┬────────────┐
│ uid ┆ timestamp ┆ item_id ┆ is_organic │
│ --- ┆ ---       ┆ ---     ┆ ---        │
│ u32 ┆ u32       ┆ u32     ┆ u8         │
╞═════╪═══════════╪═════════╪════════════╡
│ 100 ┆ 44755     ┆ 732449  ┆ 1          │
│ 100 ┆ 1155860   ┆ 6568592 ┆ 0          │
│ 100 ┆ 1259125   ┆ 5411243 ┆ 1          │
│ 100 ┆ 1260005   ┆ 7371186 ┆ 0          │
│ 100 ┆ 1263935   ┆ 4943655 ┆ 0          │
└─────┴───────────┴─────────┴────────────┘
after dedup: (844690, 3)


## 2. Filtering

In [3]:
# Item-core filter
item_counts = interactions.group_by("item_id").len().rename({"len": "cnt"})
good_items = item_counts.filter(pl.col("cnt") >= CFG["min_item_interactions"]).select("item_id")
interactions = interactions.join(good_items, on="item_id", how="semi")
print("after item-core:", interactions.shape)

# User-core filter
user_counts = interactions.group_by("uid").len().rename({"len": "cnt"})
good_users = user_counts.filter(pl.col("cnt") >= CFG["min_user_interactions"]).select("uid")
interactions = interactions.join(good_users, on="uid", how="semi")
print("after user-core:", interactions.shape)

# Optional debug subset by users. This does not break histories.
if CFG["max_users"] is not None:
    users_sub = (
        interactions
        .select("uid")
        .unique()
        .sample(n=int(CFG["max_users"]), seed=CFG["seed"])
    )
    interactions = interactions.join(users_sub, on="uid", how="semi")
    print(f"after max_users={CFG['max_users']}:", interactions.shape)

    # Repeat filters after subsampling.
    item_counts = interactions.group_by("item_id").len().rename({"len": "cnt"})
    good_items = item_counts.filter(pl.col("cnt") >= CFG["min_item_interactions"]).select("item_id")
    interactions = interactions.join(good_items, on="item_id", how="semi")

    user_counts = interactions.group_by("uid").len().rename({"len": "cnt"})
    good_users = user_counts.filter(pl.col("cnt") >= CFG["min_user_interactions"]).select("uid")
    interactions = interactions.join(good_users, on="uid", how="semi")

print("final filtered:", interactions.shape)
print("users:", interactions["uid"].n_unique())
print("items:", interactions["item_id"].n_unique())


after item-core: (621417, 3)
after user-core: (620105, 3)
final filtered: (620105, 3)
users: 7102
items: 33145


## 3. ID mapping and leave-one-out split

In [4]:
# User/item ids -> contiguous indices.
user_mapping = interactions.select("uid").unique().sort("uid").with_row_index(name="u_idx")
item_mapping = interactions.select("item_id").unique().sort("item_id").with_row_index(name="i_idx")

inter = (
    interactions
    .join(user_mapping, on="uid", how="inner")
    .join(item_mapping, on="item_id", how="inner")
    .select(["u_idx", "i_idx", "timestamp"])
    .sort(["u_idx", "timestamp"])
)

NUM_USERS = user_mapping.height
NUM_ITEMS = item_mapping.height

inter = inter.with_columns([
    pl.arange(0, pl.len()).over("u_idx").alias("pos"),
    pl.len().over("u_idx").alias("hist_len"),
])

# Leave-one-out evaluation with validation:
# most recent item -> test, second most recent -> validation, rest -> train.
train_df = inter.filter(pl.col("pos") < pl.col("hist_len") - 2).select(["u_idx", "i_idx"])
val_df   = inter.filter(pl.col("pos") == pl.col("hist_len") - 2).select(["u_idx", "i_idx"])
test_df  = inter.filter(pl.col("pos") == pl.col("hist_len") - 1).select(["u_idx", "i_idx"])

train_pairs = train_df.to_numpy().astype(np.int64)
val_np = val_df.to_numpy().astype(np.int64)
test_np = test_df.to_numpy().astype(np.int64)

val_u, val_i = val_np[:, 0], val_np[:, 1]
test_u, test_i = test_np[:, 0], test_np[:, 1]

print(f"NUM_USERS={NUM_USERS:,}")
print(f"NUM_ITEMS={NUM_ITEMS:,}")
print(f"train={len(train_pairs):,} val={len(val_u):,} test={len(test_u):,}")
print(f"catalog share @50 = {50 / NUM_ITEMS:.6f}")

assert len(train_pairs) > 0
assert len(val_u) == NUM_USERS
assert len(test_u) == NUM_USERS


NUM_USERS=7,102
NUM_ITEMS=33,145
train=605,901 val=7,102 test=7,102
catalog share @50 = 0.001509


## 4. Build item features: artist_id and album_id

In [5]:
item_features_df = item_mapping.select(["item_id", "i_idx"])

# ---------- artist feature ----------
artists = pl.read_parquet(ARTIST_PATH)
artists = normalize_item_side_columns(artists)

print("artists shape:", artists.shape)
print("artists columns:", artists.columns)

if "item_id" not in artists.columns or "artist_id" not in artists.columns:
    raise ValueError(f"Bad artist mapping columns: {artists.columns}")

# If an item has multiple artists, take the first one for a simple baseline.
artists_one = (
    artists
    .select(["item_id", "artist_id"])
    .drop_nulls(["item_id", "artist_id"])
    .group_by("item_id")
    .agg(pl.col("artist_id").first())
)

item_features_df = item_features_df.join(artists_one, on="item_id", how="left")


# ---------- album feature ----------
albums = pl.read_parquet(ALBUM_PATH)
albums = normalize_item_side_columns(albums)

print("albums shape:", albums.shape)
print("albums columns:", albums.columns)

if "item_id" not in albums.columns or "album_id" not in albums.columns:
    raise ValueError(f"Bad album mapping columns: {albums.columns}")

# If an item has multiple albums, take the first one for a simple baseline.
albums_one = (
    albums
    .select(["item_id", "album_id"])
    .drop_nulls(["item_id", "album_id"])
    .group_by("item_id")
    .agg(pl.col("album_id").first())
)

item_features_df = item_features_df.join(albums_one, on="item_id", how="left")


# ---------- categorical encoding ----------
item_features_df = item_features_df.sort("i_idx")

# unknown = 0, real categories = 1..N
unique_artists = (
    item_features_df
    .select("artist_id")
    .drop_nulls()
    .unique()
    .sort("artist_id")
    .with_row_index(name="artist_idx", offset=1)
)

item_features_df = (
    item_features_df
    .join(unique_artists, on="artist_id", how="left")
    .with_columns(pl.col("artist_idx").fill_null(0).cast(pl.Int64))
)

unique_albums = (
    item_features_df
    .select("album_id")
    .drop_nulls()
    .unique()
    .sort("album_id")
    .with_row_index(name="album_idx", offset=1)
)

item_features_df = (
    item_features_df
    .join(unique_albums, on="album_id", how="left")
    .with_columns(pl.col("album_idx").fill_null(0).cast(pl.Int64))
)

ITEM_ARTIST = item_features_df["artist_idx"].to_numpy().astype(np.int64)
ITEM_ALBUM = item_features_df["album_idx"].to_numpy().astype(np.int64)

NUM_ARTISTS = int(ITEM_ARTIST.max()) + 1
NUM_ALBUMS = int(ITEM_ALBUM.max()) + 1

print("NUM_ITEMS:", NUM_ITEMS)
print("NUM_ARTISTS:", NUM_ARTISTS)
print("NUM_ALBUMS:", NUM_ALBUMS)
print("artist unknown share:", float((ITEM_ARTIST == 0).mean()))
print("album unknown share:", float((ITEM_ALBUM == 0).mean()))

item_artist_t = torch.from_numpy(ITEM_ARTIST).long().to(device)
item_album_t = torch.from_numpy(ITEM_ALBUM).long().to(device)


artists shape: (9271906, 2)
artists columns: ['artist_id', 'item_id']
albums shape: (9651644, 2)
albums columns: ['album_id', 'item_id']
NUM_ITEMS: 33145
NUM_ARTISTS: 9321
NUM_ALBUMS: 23839
artist unknown share: 0.0
album unknown share: 3.017046311660884e-05


## 5. Known items and head/tail split

In [6]:
train_user_items = [set() for _ in range(NUM_USERS)]

for u, i in train_pairs:
    train_user_items[int(u)].add(int(i))

known_val = [set(s) for s in train_user_items]
known_test = [set(s) for s in train_user_items]

# For test, validation item is already known and should be masked.
for u, i in zip(val_u, val_i):
    known_test[int(u)].add(int(i))

# Head/tail is computed only from train frequencies.
item_freq = np.bincount(train_pairs[:, 1], minlength=NUM_ITEMS)
order = np.argsort(-item_freq)

n_head = max(1, int(NUM_ITEMS * CFG["head_fraction"]))
head_mask = np.zeros(NUM_ITEMS, dtype=bool)
head_mask[order[:n_head]] = True

print(f"head={head_mask.sum():,} tail={(~head_mask).sum():,}")
print(f"nonzero train items={np.sum(item_freq > 0):,}")
print(f"zero train items={np.sum(item_freq == 0):,}")
print(f"val true head share={head_mask[val_i].mean():.4f}")
print(f"test true head share={head_mask[test_i].mean():.4f}")


head=6,629 tail=26,516
nonzero train items=33,145
zero train items=0
val true head share=0.5596
test true head share=0.5370


## 6. Dataset and model

In [7]:
class PairDataset(Dataset):
    def __init__(self, pairs: np.ndarray):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        u, i = self.pairs[idx]
        return int(u), int(i)


class MLP(nn.Module):
    def __init__(self, in_dim: int, hidden: list[int], dropout: float = 0.0):
        super().__init__()

        layers = []
        d = in_dim

        for h in hidden:
            layers.append(nn.Linear(d, h))
            layers.append(nn.ReLU())

            if dropout > 0:
                layers.append(nn.Dropout(dropout))

            d = h

        self.net = nn.Sequential(*layers)
        self.out_dim = d

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class TwoTower(nn.Module):
    """
    Two-Tower baseline:
      user tower: user_id -> MLP
      item tower: item_id + artist_id + album_id -> MLP
    """
    def __init__(
        self,
        num_users: int,
        num_items: int,
        num_artists: int,
        num_albums: int,
        embed_dim: int = 64,
        artist_emb_dim: int = 32,
        album_emb_dim: int = 32,
        hidden: list[int] = [256, 128, 64],
        dropout: float = 0.0,
    ):
        super().__init__()

        self.user_emb = nn.Embedding(num_users, embed_dim)

        self.item_emb = nn.Embedding(num_items, embed_dim)
        self.artist_emb = nn.Embedding(num_artists, artist_emb_dim)
        self.album_emb = nn.Embedding(num_albums, album_emb_dim)

        user_in_dim = embed_dim
        item_in_dim = embed_dim + artist_emb_dim + album_emb_dim

        self.user_mlp = MLP(user_in_dim, hidden, dropout)
        self.item_mlp = MLP(item_in_dim, hidden, dropout)

        self.user_ln = nn.LayerNorm(self.user_mlp.out_dim)
        self.item_ln = nn.LayerNorm(self.item_mlp.out_dim)

        self.init_weights()

    def init_weights(self):
        for emb in [self.user_emb, self.item_emb, self.artist_emb, self.album_emb]:
            nn.init.normal_(emb.weight, std=0.01)

        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def user_vec(self, u: torch.Tensor) -> torch.Tensor:
        x = self.user_emb(u)
        x = self.user_mlp(x)
        x = self.user_ln(x)
        return x

    def item_vec(self, i: torch.Tensor) -> torch.Tensor:
        item_id_vec = self.item_emb(i)
        artist_vec = self.artist_emb(item_artist_t[i])
        album_vec = self.album_emb(item_album_t[i])

        x = torch.cat([item_id_vec, artist_vec, album_vec], dim=-1)
        x = self.item_mlp(x)
        x = self.item_ln(x)
        return x

    def forward(self, u: torch.Tensor, i: torch.Tensor) -> torch.Tensor:
        uv = F.normalize(self.user_vec(u), dim=-1, eps=1e-6)
        iv = F.normalize(self.item_vec(i), dim=-1, eps=1e-6)
        return (uv * iv).sum(dim=-1)


def inbatch_softmax_loss(user_vecs: torch.Tensor, item_vecs: torch.Tensor):
    u = F.normalize(user_vecs, dim=-1, eps=1e-6)
    v = F.normalize(item_vecs, dim=-1, eps=1e-6)

    logits = u @ v.T
    labels = torch.arange(logits.size(0), device=logits.device)

    return F.cross_entropy(logits, labels)


train_loader = DataLoader(
    PairDataset(train_pairs),
    batch_size=CFG["batch_size"],
    shuffle=True,
    drop_last=True,
    pin_memory=torch.cuda.is_available(),
)

model = TwoTower(
    NUM_USERS,
    NUM_ITEMS,
    NUM_ARTISTS,
    NUM_ALBUMS,
    embed_dim=CFG["embed_dim"],
    artist_emb_dim=CFG["artist_emb_dim"],
    album_emb_dim=CFG["album_emb_dim"],
    hidden=CFG["tower_hidden"],
    dropout=CFG["dropout"],
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=CFG["lr"],
    weight_decay=CFG["weight_decay"],
)

print(model)


TwoTower(
  (user_emb): Embedding(7102, 64)
  (item_emb): Embedding(33145, 64)
  (artist_emb): Embedding(9321, 32)
  (album_emb): Embedding(23839, 32)
  (user_mlp): MLP(
    (net): Sequential(
      (0): Linear(in_features=64, out_features=256, bias=True)
      (1): ReLU()
      (2): Linear(in_features=256, out_features=128, bias=True)
      (3): ReLU()
      (4): Linear(in_features=128, out_features=64, bias=True)
      (5): ReLU()
    )
  )
  (item_mlp): MLP(
    (net): Sequential(
      (0): Linear(in_features=128, out_features=256, bias=True)
      (1): ReLU()
      (2): Linear(in_features=256, out_features=128, bias=True)
      (3): ReLU()
      (4): Linear(in_features=128, out_features=64, bias=True)
      (5): ReLU()
    )
  )
  (user_ln): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  (item_ln): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
)


## 7. Evaluation

In [8]:
@torch.no_grad()
def evaluate_full_corpus(
    model: nn.Module,
    users: np.ndarray,
    true_items: np.ndarray,
    known_user_items: List[set],
    head_mask: np.ndarray,
    ks: List[int],
    item_freq: np.ndarray,
    user_batch_size: int = 128,
    item_batch_size: int = 8192,
):
    model.eval()

    ranks_all = []
    ranks_head = []
    ranks_tail = []

    max_k = max(ks)

    # Для coverage / popularity / personalization
    recommended_by_k = {k: [] for k in ks}

    # ============================================================
    # 1. Считаем вектора всех items один раз
    # ============================================================

    item_vectors = []

    for s in tqdm(
        range(0, NUM_ITEMS, item_batch_size),
        desc="item vectors",
        leave=False,
    ):
        e = min(s + item_batch_size, NUM_ITEMS)
        idx = torch.arange(s, e, device=device)

        v = model.item_vec(idx)
        v = F.normalize(v, dim=-1, eps=1e-6)

        if not torch.isfinite(v).all():
            raise RuntimeError(f"Non-finite item vectors on item batch {s}:{e}")

        item_vectors.append(v.cpu())

    item_vectors = torch.cat(item_vectors, dim=0).to(device)

    if not torch.isfinite(item_vectors).all():
        raise RuntimeError("Non-finite item vectors after concat")

    # ============================================================
    # 2. Идём по пользователям батчами
    # ============================================================

    for start in tqdm(
        range(0, len(users), user_batch_size),
        desc="eval users",
        leave=False,
    ):
        end = min(start + user_batch_size, len(users))

        bu = users[start:end]
        bi = true_items[start:end]

        ut = torch.tensor(bu, device=device, dtype=torch.long)

        uvec = model.user_vec(ut)
        uvec = F.normalize(uvec, dim=-1, eps=1e-6)

        if not torch.isfinite(uvec).all():
            raise RuntimeError(f"Non-finite user vectors on user batch {start}:{end}")

        scores = (uvec @ item_vectors.T).cpu().numpy()

        if not np.isfinite(scores).all():
            bad = np.sum(~np.isfinite(scores))
            raise RuntimeError(
                f"Non-finite scores in user batch {start}:{end}: {bad} bad values"
            )

        # ========================================================
        # 3. Для каждого пользователя считаем rank и top-K
        # ========================================================

        for row, (u, true_i) in enumerate(zip(bu, bi)):
            u = int(u)
            true_i = int(true_i)

            srow = scores[row].copy()

            if true_i < 0 or true_i >= NUM_ITEMS:
                raise RuntimeError(
                    f"true_i out of bounds: user={u}, true_i={true_i}, NUM_ITEMS={NUM_ITEMS}"
                )

            if not np.isfinite(srow[true_i]):
                raise RuntimeError(
                    f"Non-finite true item score: user={u}, item={true_i}"
                )

            # Маскируем уже известные items пользователя
            for it in known_user_items[u]:
                if it != true_i:
                    srow[it] = -1e9

            if not np.isfinite(srow[true_i]):
                raise RuntimeError(
                    f"True item was masked or became non-finite: user={u}, item={true_i}"
                )

            # ----------------------------------------------------
            # Rank true item
            # pessimistic tie handling:
            # если у многих items одинаковый score, true item НЕ считается первым
            # ----------------------------------------------------

            true_score = srow[true_i]
            eps = 1e-12

            num_greater = int((srow > true_score + eps).sum())
            num_tied = int((np.abs(srow - true_score) <= eps).sum())

            rank = num_greater + num_tied - 1

            ranks_all.append(rank)

            if head_mask[true_i]:
                ranks_head.append(rank)
            else:
                ranks_tail.append(rank)

            # ----------------------------------------------------
            # Top-K recommendations для coverage/popularity
            # ----------------------------------------------------

            if max_k < len(srow):
                top_unsorted = np.argpartition(-srow, max_k - 1)[:max_k]
                top_idx = top_unsorted[np.argsort(-srow[top_unsorted])]
            else:
                top_idx = np.argsort(-srow)

            for k in ks:
                recommended_by_k[k].append(top_idx[:k].astype(np.int64))

    # ============================================================
    # 4. Accuracy metrics: HR / NDCG
    # ============================================================

    def agg_accuracy(ranks: List[int]) -> Dict[str, float]:
        a = np.asarray(ranks, dtype=np.int64)
        out = {}

        for k in ks:
            if len(a) == 0:
                out[f"HR@{k}"] = np.nan
                out[f"NDCG@{k}"] = np.nan
            else:
                hits = a < k

                out[f"HR@{k}"] = 100.0 * hits.mean()

                out[f"NDCG@{k}"] = 100.0 * np.where(
                    hits,
                    1.0 / np.log2(a + 2),
                    0.0,
                ).mean()

        return out

    # ============================================================
    # 5. Long-tail / coverage metrics
    # ============================================================

    tail_mask = ~head_mask
    num_tail_items = int(tail_mask.sum())

    # item_freq нужен именно здесь
    popularity = np.log1p(item_freq.astype(np.float64))

    long_tail_metrics = {}

    for k in ks:
        recs = np.vstack(recommended_by_k[k])  # shape: (num_eval_users, k)

        unique_recs = np.unique(recs)

        # CatalogCoverage@K
        catalog_coverage = len(unique_recs) / NUM_ITEMS

        # TailCoverage@K
        if num_tail_items > 0:
            tail_coverage = np.sum(tail_mask[unique_recs]) / num_tail_items
        else:
            tail_coverage = np.nan

        # AvgPopularity@K
        avg_popularity = popularity[recs].mean()

        # TailRatio@K
        tail_ratio = tail_mask[recs].mean()

        # Personalization@K
        # 1 - average pairwise overlap / K
        # считаем эффективно через exposure counts
        n_users_eval = recs.shape[0]

        if n_users_eval <= 1:
            personalization = np.nan
        else:
            exposure_counts = np.bincount(
                recs.reshape(-1),
                minlength=NUM_ITEMS,
            )

            pairwise_intersections = np.sum(
                exposure_counts * (exposure_counts - 1) / 2
            )

            num_user_pairs = n_users_eval * (n_users_eval - 1) / 2
            avg_overlap = pairwise_intersections / num_user_pairs

            personalization = 1.0 - avg_overlap / k

        long_tail_metrics[f"CatalogCoverage@{k}"] = 100.0 * catalog_coverage
        long_tail_metrics[f"TailCoverage@{k}"] = 100.0 * tail_coverage
        long_tail_metrics[f"AvgPopularity@{k}"] = float(avg_popularity)
        long_tail_metrics[f"TailRatio@{k}"] = 100.0 * tail_ratio
        long_tail_metrics[f"Personalization@{k}"] = 100.0 * personalization

    return {
        "overall": agg_accuracy(ranks_all),
        "head": agg_accuracy(ranks_head),
        "tail": agg_accuracy(ranks_tail),
        "long_tail": long_tail_metrics,
        "counts": {
            "overall": len(ranks_all),
            "head": len(ranks_head),
            "tail": len(ranks_tail),
        },
    }


def print_metrics(metrics: Dict):
    print("counts:", metrics.get("counts", {}))

    for split in ["overall", "head", "tail"]:
        print(f"[{split}]", metrics[split])

    if "long_tail" in metrics:
        print("[long_tail]", metrics["long_tail"])

In [9]:
def make_head_mask(item_freq: np.ndarray, head_fraction: float) -> np.ndarray:
    num_items = len(item_freq)
    n_head = max(1, int(num_items * head_fraction))

    order = np.argsort(-item_freq)

    current_head_mask = np.zeros(num_items, dtype=bool)
    current_head_mask[order[:n_head]] = True

    return current_head_mask


@torch.no_grad()
def evaluate_head_tail_sweep(
    model: nn.Module,
    method_name: str,
    seed: int,
    head_fractions: List[float],
    test_u: np.ndarray,
    test_i: np.ndarray,
    known_test: List[set],
    item_freq: np.ndarray,
    ks: List[int],
):
    rows = []

    model.eval()

    for hf in head_fractions:
        print("\n" + "=" * 80)
        print(f"{method_name} | seed={seed} | head_fraction={hf} ({100 * hf:.3f}%)")
        print("=" * 80)

        current_head_mask = make_head_mask(
            item_freq=item_freq,
            head_fraction=hf,
        )

        num_head_items = int(current_head_mask.sum())
        num_tail_items = int((~current_head_mask).sum())

        test_head_share = float(current_head_mask[test_i].mean())
        test_tail_share = float((~current_head_mask[test_i]).mean())

        print(f"num_head_items: {num_head_items:,}")
        print(f"num_tail_items: {num_tail_items:,}")
        print(f"test_head_share: {test_head_share:.4f}")
        print(f"test_tail_share: {test_tail_share:.4f}")

        metrics = evaluate_full_corpus(
            model=model,
            users=test_u,
            true_items=test_i,
            known_user_items=known_test,
            head_mask=current_head_mask,
            ks=ks,
            item_freq=item_freq,
            user_batch_size=CFG["eval_batch_users"],
            item_batch_size=CFG["eval_item_batch"],
        )

        print_metrics(metrics)

        row = {
            "method": method_name,
            "seed": seed,
            "head_fraction": hf,
            "head_percent": 100.0 * hf,
            "num_head_items": num_head_items,
            "num_tail_items": num_tail_items,
            "test_head_share": test_head_share,
            "test_tail_share": test_tail_share,
        }

        for split in ("overall", "head", "tail"):
            for key, value in metrics[split].items():
                row[f"{split}_{key}"] = value

        if "long_tail" in metrics:
            for key, value in metrics["long_tail"].items():
                row[key] = value

        if "counts" in metrics:
            for key, value in metrics["counts"].items():
                row[f"count_{key}"] = value

        rows.append(row)

    return rows

## 8. Grid search

In [10]:
LR_GRID = [0.001]
DROPOUT_GRID = [0.1]
WD_GRID = [0.0]

combos = list(itertools.product(LR_GRID, DROPOUT_GRID, WD_GRID))
k_main = CFG["eval_k"][-1]  # HR@50

cache_path = Path(CFG["cache_path"])
leaderboard_csv_path = cache_path.with_suffix(".leaderboard.csv")

if CFG["skip_tune_if_cached"] and cache_path.exists():
    best_hp = json.loads(cache_path.read_text())
    print(f"Loaded cached hparams: {best_hp}")

    if leaderboard_csv_path.exists():
        leaderboard_df = pd.read_csv(leaderboard_csv_path)
        print(f"Loaded leaderboard: {leaderboard_csv_path}")
    else:
        leaderboard_df = None
        print("Leaderboard CSV not found.")

else:
    frac = float(CFG.get("tune_val_fraction", 1.0))

    if frac < 1.0 and CFG["tune_fast"]:
        n_tune = max(1, int(len(val_u) * frac))
        _idx = np.random.default_rng(42).choice(len(val_u), n_tune, replace=False)
        val_u_t, val_i_t = val_u[_idx], val_i[_idx]
    else:
        val_u_t, val_i_t = val_u, val_i

    tune_ep = CFG["tune_epochs"] if CFG["tune_fast"] else CFG["final_epochs"]

    print(
        f"Tuning {len(combos)} trials × {tune_ep} ep "
        f"(val subset: {len(val_u_t):,}/{len(val_u):,})"
    )

    best_hr = -1.0
    best_hp = None
    leaderboard = []

    for ti, (lr, dr, wd) in enumerate(combos, 1):
        set_seed(CFG["seed"])

        m = TwoTower(
            NUM_USERS,
            NUM_ITEMS,
            NUM_ARTISTS,
            NUM_ALBUMS,
            embed_dim=CFG["embed_dim"],
            artist_emb_dim=CFG["artist_emb_dim"],
            album_emb_dim=CFG["album_emb_dim"],
            hidden=CFG["tower_hidden"],
            dropout=dr,
        ).to(device)

        opt = torch.optim.Adam(
            m.parameters(),
            lr=lr,
            weight_decay=wd,
        )

        status = "ok"
        error_message = ""
        avg_loss = np.nan
        met = None
        hr = -1.0

        try:
            for ep in range(1, tune_ep + 1):
                m.train()
                total_loss = 0.0
                total_n = 0

                pbar = tqdm(
                    train_loader,
                    desc=f"trial{ti} {ep}/{tune_ep}",
                    leave=False,
                )

                for users_batch, items_batch in pbar:
                    users_batch = users_batch.to(device, non_blocking=True)
                    items_batch = items_batch.to(device, non_blocking=True)

                    user_vecs = m.user_vec(users_batch)
                    item_vecs = m.item_vec(items_batch)

                    loss = inbatch_softmax_loss(user_vecs, item_vecs)

                    if not torch.isfinite(loss):
                        raise RuntimeError(f"Non-finite loss: {loss.item()}")

                    opt.zero_grad(set_to_none=True)
                    loss.backward()

                    torch.nn.utils.clip_grad_norm_(
                        m.parameters(),
                        CFG["grad_clip"],
                    )

                    opt.step()

                    bs = users_batch.size(0)
                    total_loss += loss.item() * bs
                    total_n += bs

                    pbar.set_postfix(loss=f"{loss.item():.4f}")

                avg_loss = total_loss / max(total_n, 1)
                print(f"  trial{ti} ep{ep} loss={avg_loss:.4f}")

            met = evaluate_full_corpus(
                model=m,
                users=val_u_t,
                true_items=val_i_t,
                known_user_items=known_val,
                head_mask=head_mask,
                ks=CFG["eval_k"],
                item_freq=item_freq,
                user_batch_size=CFG["eval_batch_users"],
                item_batch_size=CFG["eval_item_batch"],
            )

            hr = met["overall"][f"HR@{k_main}"]

            if not np.isfinite(hr):
                raise RuntimeError(f"Non-finite HR: {hr}")

        except RuntimeError as e:
            status = "failed"
            error_message = str(e)
            print(f"  trial {ti} FAILED: {e}")

        row = {
            "trial": ti,
            "status": status,
            "error": error_message,
            "lr": lr,
            "dropout": dr,
            "weight_decay": wd,
            "tune_epochs": tune_ep,
            "val_subset_size": len(val_u_t),
            "val_full_size": len(val_u),
            "final_train_loss": avg_loss,
            f"val_HR@{k_main}": hr,
        }

        if met is not None:
            for split in ("overall", "head", "tail"):
                for key, value in met[split].items():
                    row[f"val_{split}_{key}"] = value

            if "long_tail" in met:
                for key, value in met["long_tail"].items():
                    row[f"val_{key}"] = value

            if "counts" in met:
                for key, value in met["counts"].items():
                    row[f"val_count_{key}"] = value

        leaderboard.append(row)

        print(
            f"  trial {ti:3d}/{len(combos)}  "
            f"lr={lr:.0e} dr={dr} wd={wd:.0e}  "
            f"-> val HR@{k_main}={hr:.2f}%"
        )

        if status == "ok" and hr > best_hr:
            best_hr = hr
            best_hp = {
                "lr": lr,
                "dropout": dr,
                "weight_decay": wd,
                f"val_HR@{k_main}": hr,
            }

        del m
        del opt
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    leaderboard_df = pd.DataFrame(leaderboard).sort_values(
        f"val_HR@{k_main}",
        ascending=False,
        na_position="last",
    )

    leaderboard_df.to_csv(leaderboard_csv_path, index=False)

    if best_hp is None:
        raise RuntimeError("No successful grid-search trial. Check leaderboard for errors.")

    cache_path.write_text(json.dumps(best_hp, indent=2))

    print(f"\nBest val HR@{k_main}={best_hr:.2f}%  ->  {best_hp}")
    print(f"Saved best hparams: {cache_path}")
    print(f"Saved leaderboard CSV: {leaderboard_csv_path}")

leaderboard_df.head(10) if leaderboard_df is not None else None

Tuning 80 trials × 3 ep (val subset: 7,102/7,102)


  trial1 ep1 loss=8.3178


  trial1 ep2 loss=8.2863


  trial1 ep3 loss=8.1143


  trial   1/80  lr=1e-01 dr=0.1 wd=0e+00  -> val HR@50=0.27%


...

  trial80 ep1 loss=8.3272


  trial80 ep2 loss=8.3266


  trial80 ep3 loss=8.3256


  trial  80/80  lr=1e-04 dr=0.9 wd=1e-03  -> val HR@50=0.39%

Best val HR@50=0.54%  ->  {'lr': 0.001, 'dropout': 0.9, 'weight_decay': 0.001, 'val_HR@50': 0.5350605463249789}
Saved best hparams: best_hparams_yambda_likes_features.json
Saved leaderboard CSV: best_hparams_yambda_likes_features.leaderboard.csv


,trial,status,error,lr,dropout,weight_decay,tune_epochs,val_subset_size,val_full_size,final_train_loss,...,val_TailRatio@10,val_Personalization@10,val_CatalogCoverage@50,val_TailCoverage@50,val_AvgPopularity@50,val_TailRatio@50,val_Personalization@50,val_count_overall,val_count_head,val_count_tail
62,63,ok,,0.0001,0.1,0.010,3,7102,7102,8.316529,...,44.645170,83.326396,3.780359,3.205612,2.991524,61.610532,81.245740,7102,3974,3128
59,60,ok,,0.0010,0.9,0.001,3,7102,7102,8.317935,...,50.901155,1.821023,0.229296,0.169709,3.248158,52.811321,1.672336,7102,3974,3128
78,79,ok,,0.0001,0.9,0.010,3,7102,7102,8.322457,...,40.143622,2.314196,0.208176,0.139538,3.427526,48.813855,1.849968,7102,3974,3128
40,41,ok,,0.0010,0.1,0.000,3,7102,7102,8.055319,...,78.687694,99.817189,79.526324,79.054156,2.593676,77.902844,99.614877,7102,3974,3128
63,64,ok,,0.0001,0.1,0.001,3,7102,7102,8.233377,...,80.361870,99.669102,83.542012,83.225223,2.556545,80.294002,99.301153,7102,3974,3128
79,80,ok,,0.0001,0.9,0.001,3,7102,7102,8.325608,...,32.899183,44.088285,2.066677,1.674461,3.058578,50.826809,38.365570,7102,3974,3128
74,75,ok,,0.0001,0.7,0.010,3,7102,7102,8.317879,...,60.004224,1.272647,0.211193,0.158395,3.146612,56.170375,1.327517,7102,3974,3128
20,21,ok,,0.0100,0.1,0.000,3,7102,7102,8.069846,...,85.387215,97.410583,34.119777,34.311359,2.496284,82.497043,98.298972,7102,3974,3128
71,72,ok,,0.0001,0.5,0.001,3,7102,7102,8.317890,...,70.264714,0.722323,0.178006,0.139538,2.998956,62.249507,0.824661,7102,3974,3128
70,71,ok,,0.0001,0.5,0.010,3,7102,7102,8.317851,...,50.418192,1.049374,0.271534,0.214965,3.090882,60.171219,1.020984,7102,3974,3128


## 9. Final training

In [11]:
HEAD_FRACTIONS = [0.001, 0.005, 0.01, 0.05, 0.20]

all_rows = []
all_test = []
all_sweep_rows = []

print("Using best_hp:", best_hp)

for seed in CFG["seeds"]:
    print("\n" + "=" * 80)
    print(f"TwoTower seed {seed}")
    print("=" * 80)

    set_seed(seed)

    model = TwoTower(
        NUM_USERS,
        NUM_ITEMS,
        NUM_ARTISTS,
        NUM_ALBUMS,
        embed_dim=CFG["embed_dim"],
        artist_emb_dim=CFG["artist_emb_dim"],
        album_emb_dim=CFG["album_emb_dim"],
        hidden=CFG["tower_hidden"],
        dropout=best_hp["dropout"],
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=best_hp["lr"],
        weight_decay=best_hp["weight_decay"],
    )

    best_val_hr50 = -1.0
    best_state = None

    # -------------------------
    # Train
    # -------------------------
    for epoch in range(1, CFG["final_epochs"] + 1):
        model.train()
        total_loss = 0.0
        total_n = 0

        pbar = tqdm(
            train_loader,
            desc=f"seed {seed} train {epoch}/{CFG['final_epochs']}",
            leave=False,
        )

        for users_batch, items_batch in pbar:
            users_batch = users_batch.to(device, non_blocking=True)
            items_batch = items_batch.to(device, non_blocking=True)

            user_vecs = model.user_vec(users_batch)
            item_vecs = model.item_vec(items_batch)

            loss = inbatch_softmax_loss(user_vecs, item_vecs)

            if not torch.isfinite(loss):
                raise RuntimeError(
                    f"Non-finite loss at seed={seed}, epoch={epoch}: {loss.item()}"
                )

            optimizer.zero_grad(set_to_none=True)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                CFG["grad_clip"],
            )

            optimizer.step()

            bs = users_batch.size(0)
            total_loss += loss.item() * bs
            total_n += bs

            pbar.set_postfix(loss=f"{loss.item():.4f}")

        avg_loss = total_loss / max(total_n, 1)
        print(f"seed {seed} epoch {epoch}: train loss = {avg_loss:.4f}")

        # -------------------------
        # Validation
        # -------------------------
        val_metrics = evaluate_full_corpus(
            model=model,
            users=val_u,
            true_items=val_i,
            known_user_items=known_val,
            head_mask=head_mask,
            ks=CFG["eval_k"],
            item_freq=item_freq,
            user_batch_size=CFG["eval_batch_users"],
            item_batch_size=CFG["eval_item_batch"],
        )

        hr50 = val_metrics["overall"].get("HR@50", -1.0)

        if hr50 > best_val_hr50:
            best_val_hr50 = hr50
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }

            print(f"new best val HR@50 = {best_val_hr50:.4f}")

    print(f"seed {seed} best val HR@50: {best_val_hr50:.4f}")

    assert best_state is not None

    model.load_state_dict(best_state)
    model.to(device)
    model.eval()

    # -------------------------
    # Test on default head_mask
    # -------------------------
    test_metrics = evaluate_full_corpus(
        model=model,
        users=test_u,
        true_items=test_i,
        known_user_items=known_test,
        head_mask=head_mask,
        ks=CFG["eval_k"],
        item_freq=item_freq,
        user_batch_size=CFG["eval_batch_users"],
        item_batch_size=CFG["eval_item_batch"],
    )

    print("TEST METRICS")
    print_metrics(test_metrics)

    all_test.append(test_metrics)

    row = {
        "method": "TwoTower",
        "seed": seed,
        "lr": best_hp["lr"],
        "dropout": best_hp["dropout"],
        "weight_decay": best_hp["weight_decay"],
        "best_val_HR@50": best_val_hr50,
    }

    for split in ("overall", "head", "tail"):
        for key, value in test_metrics[split].items():
            row[f"test_{split}_{key}"] = value

    if "long_tail" in test_metrics:
        for key, value in test_metrics["long_tail"].items():
            row[f"test_{key}"] = value

    if "counts" in test_metrics:
        for key, value in test_metrics["counts"].items():
            row[f"test_count_{key}"] = value

    all_rows.append(row)

    # -------------------------
    # Head/tail threshold sweep
    # -------------------------
    sweep_rows_seed = evaluate_head_tail_sweep(
        model=model,
        method_name="TwoTower",
        seed=seed,
        head_fractions=HEAD_FRACTIONS,
        test_u=test_u,
        test_i=test_i,
        known_test=known_test,
        item_freq=item_freq,
        ks=CFG["eval_k"],
    )

    all_sweep_rows.extend(sweep_rows_seed)

    del model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


metrics_df = pd.DataFrame(all_rows)
sweep_df = pd.DataFrame(all_sweep_rows)

metrics_df

Using best_hp: {'lr': 0.001, 'dropout': 0.9, 'weight_decay': 0.001, 'val_HR@50': 0.5350605463249789}

TwoTower seed 0


seed 0 epoch 1: train loss = 8.3241


new best val HR@50 = 0.9716


seed 0 epoch 2: train loss = 8.3187


seed 0 epoch 3: train loss = 8.3179


...

counts: {'overall': 7102, 'head': 459, 'tail': 6643}
[overall] {'HR@10': 0.0, 'NDCG@10': 0.0, 'HR@50': 0.5632216277105041, 'NDCG@50': 0.10087784837369185}
[head] {'HR@10': 0.0, 'NDCG@10': 0.0, 'HR@50': 5.010893246187364, 'NDCG@50': 0.8939527951977984}
[tail] {'HR@10': 0.0, 'NDCG@10': 0.0, 'HR@50': 0.2559084750865573, 'NDCG@50': 0.04608010630049227}
[long_tail] {'CatalogCoverage@10': 0.5098808266706895, 'TailCoverage@10': 0.5033353547604609, 'AvgPopularity@10': 3.0892006598097437, 'TailRatio@10': 97.15150661785412, 'Personalization@10': 67.81939597752206, 'CatalogCoverage@50': 1.1585457836777795, 'TailCoverage@50': 1.1522134627046696, 'AvgPopularity@50': 3.188902421384942, 'TailRatio@50': 94.5060546324979, 'Personalization@50': 6.263736438928347}

TwoTower | seed=4 | head_fraction=0.01 (1.000%)
num_head_items: 331
num_tail_items: 32,814
test_head_share: 0.1035
test_tail_share: 0.8965


counts: {'overall': 7102, 'head': 735, 'tail': 6367}
[overall] {'HR@10': 0.0, 'NDCG@10': 0.0, 'HR@50': 0.5632216277105041, 'NDCG@50': 0.10087784837369185}
[head] {'HR@10': 0.0, 'NDCG@10': 0.0, 'HR@50': 3.537414965986395, 'NDCG@50': 0.6310888093628514}
[tail] {'HR@10': 0.0, 'NDCG@10': 0.0, 'HR@50': 0.21988377571854878, 'NDCG@50': 0.039670834658122156}
[long_tail] {'CatalogCoverage@10': 0.5098808266706895, 'TailCoverage@10': 0.5028341561528615, 'AvgPopularity@10': 3.0892006598097437, 'TailRatio@10': 96.6192621796677, 'Personalization@10': 67.81939597752206, 'CatalogCoverage@50': 1.1585457836777795, 'TailCoverage@50': 1.151947339550192, 'AvgPopularity@50': 3.188902421384942, 'TailRatio@50': 92.46043368065334, 'Personalization@50': 6.263736438928347}

TwoTower | seed=4 | head_fraction=0.05 (5.000%)
num_head_items: 1,657
num_tail_items: 31,488
test_head_share: 0.2858
test_tail_share: 0.7142


counts: {'overall': 7102, 'head': 2030, 'tail': 5072}
[overall] {'HR@10': 0.0, 'NDCG@10': 0.0, 'HR@50': 0.5632216277105041, 'NDCG@50': 0.10087784837369185}
[head] {'HR@10': 0.0, 'NDCG@10': 0.0, 'HR@50': 1.7241379310344827, 'NDCG@50': 0.30839464382836934}
[tail] {'HR@10': 0.0, 'NDCG@10': 0.0, 'HR@50': 0.09858044164037855, 'NDCG@50': 0.017822033158195947}
[long_tail] {'CatalogCoverage@10': 0.5098808266706895, 'TailCoverage@10': 0.4795477642276422, 'AvgPopularity@10': 3.0892006598097437, 'TailRatio@10': 75.32807659814137, 'Personalization@10': 67.81939597752206, 'CatalogCoverage@50': 1.1585457836777795, 'TailCoverage@50': 1.1369410569105691, 'AvgPopularity@50': 3.188902421384942, 'TailRatio@50': 72.7105040833568, 'Personalization@50': 6.263736438928347}

TwoTower | seed=4 | head_fraction=0.2 (20.000%)
num_head_items: 6,629
num_tail_items: 26,516
test_head_share: 0.5370
test_tail_share: 0.4630


counts: {'overall': 7102, 'head': 3814, 'tail': 3288}
[overall] {'HR@10': 0.0, 'NDCG@10': 0.0, 'HR@50': 0.5632216277105041, 'NDCG@50': 0.10087784837369185}
[head] {'HR@10': 0.0, 'NDCG@10': 0.0, 'HR@50': 1.048767697954903, 'NDCG@50': 0.18784333485840574}
[tail] {'HR@10': 0.0, 'NDCG@10': 0.0, 'HR@50': 0.0, 'NDCG@50': 0.0}
[long_tail] {'CatalogCoverage@10': 0.5098808266706895, 'TailCoverage@10': 0.47895610197616534, 'AvgPopularity@10': 3.0892006598097437, 'TailRatio@10': 56.98817234581808, 'Personalization@10': 67.81939597752206, 'CatalogCoverage@50': 1.1585457836777795, 'TailCoverage@50': 1.0974505958666467, 'AvgPopularity@50': 3.188902421384942, 'TailRatio@50': 58.09124190368911, 'Personalization@50': 6.263736438928347}


,method,seed,lr,dropout,weight_decay,best_val_HR@50,test_overall_HR@10,test_overall_NDCG@10,test_overall_HR@50,test_overall_NDCG@50,...,test_TailRatio@10,test_Personalization@10,test_CatalogCoverage@50,test_TailCoverage@50,test_AvgPopularity@50,test_TailRatio@50,test_Personalization@50,test_count_overall,test_count_head,test_count_tail
0,TwoTower,0,0.001,0.9,0.001,0.971557,0.422416,0.190222,0.718108,0.259335,...,0.180231,6.434834,0.202142,0.113139,3.547882,41.972965,2.856856,7102,3814,3288
1,TwoTower,1,0.001,0.9,0.001,0.718108,0.239369,0.103350,0.549141,0.173629,...,30.033793,3.304239,0.199125,0.131996,3.354717,44.524078,3.245621,7102,3814,3288
2,TwoTower,2,0.001,0.9,0.001,0.535061,0.000000,0.000000,0.591383,0.122281,...,80.092932,2.369967,0.223261,0.135767,3.421232,44.192622,1.509719,7102,3814,3288
3,TwoTower,3,0.001,0.9,0.001,0.535061,0.000000,0.000000,0.591383,0.128930,...,59.900028,1.255352,0.247398,0.150852,3.575622,40.740636,2.056895,7102,3814,3288
4,TwoTower,4,0.001,0.9,0.001,0.577302,0.000000,0.000000,0.563222,0.100878,...,56.988172,67.819396,1.158546,1.097451,3.188902,58.091242,6.263736,7102,3814,3288


In [12]:
def make_final_summary_table(
    metrics_df: pd.DataFrame,
    sweep_df: pd.DataFrame,
    method_name: str = "TwoTower",
    save_path: str | None = None,
) -> pd.DataFrame:
    """
    Делает одну финальную таблицу mean ± std по seed-ам.

    metrics_df:
        обычные test metrics по seed-ам.
        one row = one seed.

    sweep_df:
        head/tail sweep по seed-ам.
        one row = one seed × one head_fraction.

    Возвращает:
        summary_df:
            one row = one method × one head_fraction.
            Метрики записаны в формате 'mean ± std'.
    """

    if len(sweep_df) == 0:
        raise ValueError("sweep_df is empty")

    # Метрики, которые хотим видеть в финальной таблице
    selected_metrics = [
        "overall_HR@50",
        "overall_NDCG@50",
        "head_HR@50",
        "head_NDCG@50",
        "tail_HR@50",
        "tail_NDCG@50",
        "CatalogCoverage@50",
        "TailCoverage@50",
        "AvgPopularity@50",
        "TailRatio@50",
        "Personalization@50",
    ]

    rows = []

    # groupby по head_fraction: внутри группы разные seeds
    for head_fraction, group in sweep_df.groupby("head_fraction"):
        group = group.copy()

        row = {
            "method": method_name,
            "head_share": f"{100 * float(head_fraction):.3f}%",
            "head_fraction": float(head_fraction),
            "num_seeds": group["seed"].nunique() if "seed" in group.columns else len(group),
            "num_head_items": int(group["num_head_items"].iloc[0]) if "num_head_items" in group.columns else np.nan,
            "num_tail_items": int(group["num_tail_items"].iloc[0]) if "num_tail_items" in group.columns else np.nan,
        }

        if "test_head_share" in group.columns:
            row["test_head_share"] = f"{100 * float(group['test_head_share'].mean()):.2f}%"

        if "test_tail_share" in group.columns:
            row["test_tail_share"] = f"{100 * float(group['test_tail_share'].mean()):.2f}%"

        if metrics_df is not None and len(metrics_df) > 0:
            for hp_col in ["lr", "dropout", "weight_decay", "logq_weight", "cb_beta"]:
                if hp_col in metrics_df.columns:
                    vals = metrics_df[hp_col].dropna().unique()
                    if len(vals) == 1:
                        row[hp_col] = vals[0]
                    elif len(vals) > 1:
                        row[hp_col] = ", ".join(map(str, vals))

            if "best_val_HR@50" in metrics_df.columns:
                vals = metrics_df["best_val_HR@50"].dropna().to_numpy(dtype=float)
                if len(vals) > 0:
                    mean = float(np.mean(vals))
                    std = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
                    row["best_val_HR@50"] = f"{mean:.2f} ± {std:.2f}"

        for metric in selected_metrics:
            if metric not in group.columns:
                continue

            vals = group[metric].dropna().to_numpy(dtype=float)

            if len(vals) == 0:
                row[metric] = "nan"
                continue

            mean = float(np.mean(vals))
            std = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0

            if "AvgPopularity" in metric:
                row[metric] = f"{mean:.4f} ± {std:.4f}"
            else:
                row[metric] = f"{mean:.2f} ± {std:.2f}"

        rows.append(row)

    summary_df = pd.DataFrame(rows).sort_values("head_fraction").reset_index(drop=True)

    first_cols = [
        "method",
        "head_share",
        "head_fraction",
        "num_seeds",
        "num_head_items",
        "num_tail_items",
        "test_head_share",
        "test_tail_share",
        "lr",
        "dropout",
        "weight_decay",
        "logq_weight",
        "cb_beta",
        "best_val_HR@50",
    ]

    metric_cols = [
        "overall_HR@50",
        "overall_NDCG@50",
        "head_HR@50",
        "head_NDCG@50",
        "tail_HR@50",
        "tail_NDCG@50",
        "CatalogCoverage@50",
        "TailCoverage@50",
        "AvgPopularity@50",
        "TailRatio@50",
        "Personalization@50",
    ]

    ordered_cols = [
        c for c in first_cols + metric_cols
        if c in summary_df.columns
    ]

    other_cols = [
        c for c in summary_df.columns
        if c not in ordered_cols
    ]

    summary_df = summary_df[ordered_cols + other_cols]

    if save_path is not None:
        summary_df.to_csv(save_path, index=False)
        print(f"saved: {save_path}")

    return summary_df

In [13]:
summary_df = make_final_summary_table(metrics_df, sweep_df, save_path="yambda_twotower_summary.csv")

saved: yambda_twotower_summary.csv


In [14]:
summary_df

,method,head_share,head_fraction,num_seeds,num_head_items,num_tail_items,test_head_share,test_tail_share,lr,dropout,...,overall_NDCG@50,head_HR@50,head_NDCG@50,tail_HR@50,tail_NDCG@50,CatalogCoverage@50,TailCoverage@50,AvgPopularity@50,TailRatio@50,Personalization@50
0,TwoTower,0.100%,0.001,5,33,33112,2.66%,97.34%,0.001,0.9,...,0.16 ± 0.06,10.05 ± 4.92,2.90 ± 2.09,0.34 ± 0.11,0.08 ± 0.02,0.41 ± 0.42,0.40 ± 0.42,3.4177 ± 0.1567,94.82 ± 2.77,3.19 ± 1.85
1,TwoTower,0.500%,0.005,5,165,32980,6.46%,93.54%,0.001,0.9,...,0.16 ± 0.06,4.84 ± 2.25,1.38 ± 0.95,0.31 ± 0.12,0.07 ± 0.02,0.41 ± 0.42,0.39 ± 0.42,3.4177 ± 0.1567,91.72 ± 4.77,3.19 ± 1.85
2,TwoTower,1.000%,0.010,5,331,32814,10.35%,89.65%,0.001,0.9,...,0.16 ± 0.06,3.48 ± 1.45,0.96 ± 0.55,0.27 ± 0.13,0.06 ± 0.03,0.41 ± 0.42,0.39 ± 0.43,3.4177 ± 0.1567,87.99 ± 4.59,3.19 ± 1.85
3,TwoTower,5.000%,0.050,5,1657,31488,28.58%,71.42%,0.001,0.9,...,0.16 ± 0.06,1.78 ± 0.26,0.48 ± 0.20,0.13 ± 0.04,0.03 ± 0.01,0.41 ± 0.42,0.36 ± 0.43,3.4177 ± 0.1567,70.87 ± 4.16,3.19 ± 1.85
4,TwoTower,20.000%,0.200,5,6629,26516,53.70%,46.30%,0.001,0.9,...,0.16 ± 0.06,1.09 ± 0.15,0.29 ± 0.12,0.04 ± 0.06,0.01 ± 0.01,0.41 ± 0.42,0.33 ± 0.43,3.4177 ± 0.1567,45.90 ± 6.99,3.19 ± 1.85
